# 06-phase7-amplitude-smooth-start

**neuron Phase 7** — Phase 6 의 결정적 발견 (amplitude vanishing gradient) 해소 시도. Phase 2 의
sweet spot 교훈 (α=0 init → dead block → nonzero init 으로 해결) 을 SinusoidalAlpha 의
amplitude 에 재적용.

- **Phase 6 결과** (PR #52): positional ≈ per_channel (Δ=-0.0001), amplitude 거의 안 자람 (max 0.05~0.09), log_freq 1.5% 변화
- **본 Phase 의 가설**:
  1. **amplitude smooth-start 가 vanishing 해소** — init_amp nonzero 면 amplitude 가 학습되는가?
  2. **log_freq 도 동반 학습** — amplitude 활성화 → sin 항 gradient → log_freq 도 함께?
  3. **final_loss 영향** — amplitude 학습이 final_loss 개선 / 동일 / 악화?
  4. **partial-dead 패턴 재발현** — Phase 5 의 implicit pruning 이 amplitude 활성화 후 재현?
  5. **sweet spot init_amp** — 어느 값이 최적?
- **설계**: 3 × 3 sweep (`init_amp` ∈ {0.0, 0.03, 0.10} × seed ∈ {42, 123, 456}) = 9 run, max_steps=1500.
- **데이터**: TinyShakespeare (char-LM)
- **시드**: [42, 123, 456]
- **작성일**: 2026-05-25
- **연관**: Issue [#53](https://github.com/EinSofINTEREST/GraphLM/issues/53) / Phase 6 baseline PR [#52](https://github.com/EinSofINTEREST/GraphLM/pull/52)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import (
    NeuronConfig,
    NeuronGrowingDecoder,
    SinusoidalAlpha,
    add_attn_smooth_start,
)
from graphlm.neuron.positional_analysis import eval_positional_alpha
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정 — init_amp sweep (Phase 6 모두 0.0)

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
# 0.0 = Phase 6 baseline (재현성 확인) / 0.03 = 작은 nonzero / 0.10 = bias 와 동일 스케일
INIT_AMPS = [0.0, 0.03, 0.10]
INIT_BIAS = (
    0.10  # positional alpha 의 add_attn residual_scale = init_bias 고정 (Phase 2~6 sweet spot)
)

BATCH_SIZE = 16
BLOCK_SIZE = 128
BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05
PARTIAL_DEAD_THRESH = 0.80

# Phase 6 baseline (PR #52 — positional with init_amp=0)
PHASE6_POS_MEAN = 1.7779
PHASE6_POS_SIGMA = 0.0065
PHASE6_PERCH_MEAN = 1.7778
GRAPHLM_PHASE1_4L_FINAL_LOSS = 1.7871

print(f"SEEDS        = {SEEDS}")
print(f"INIT_AMPS    = {INIT_AMPS}")
print(f"INIT_BIAS    = {INIT_BIAS} (residual_scale 로 고정)")
print(f"max_steps    = {TRAIN['max_steps']}")
print(f"전체 run     = {len(SEEDS) * len(INIT_AMPS)} (GPU 약 14분 예상)")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 3 x 3 sweep 학습

- `cfg.alpha_positional=True` + `cfg.alpha_positional_init_amp` 변경
- 그 외 Phase 6 와 동일 (fair 비교)

In [ ]:
runs = {}
for init_amp in INIT_AMPS:
    cfg = NeuronConfig(
        vocab_size=tokenizer.vocab_size,
        max_seq_len=BLOCK_SIZE,
        alpha_positional=True,
        alpha_positional_init_amp=init_amp,
        **BACKBONE,
    )
    for seed in SEEDS:
        print(f"--- init_amp={init_amp}, seed={seed} ---")
        set_seed(seed)
        model = NeuronGrowingDecoder(cfg).to(DEVICE)
        data_iter = iter_random_batches(
            dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
        )
        trigger = PlateauTrigger(
            window=TRAIN["trigger_window"],
            epsilon=TRAIN["trigger_epsilon"],
            cooldown=TRAIN["trigger_cooldown"],
            min_history=TRAIN["trigger_min_history"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

        losses = []
        grow_events = []
        next_block_to_grow = 0
        V = cfg.vocab_size
        model.train()
        for step in range(1, TRAIN["max_steps"] + 1):
            x, y = next(data_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if (
                trigger.update(loss.item())
                and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]
            ):
                target_block = next_block_to_grow
                add_attn_smooth_start(model, target_block, alpha_init=INIT_BIAS)
                block = model.blocks[target_block]
                new_alpha = block.attn_alphas[-1]
                if isinstance(new_alpha, SinusoidalAlpha):
                    new_alpha_params = list(new_alpha.parameters())
                else:
                    new_alpha_params = [new_alpha]
                new_params = (
                    list(block.attn_lns[-1].parameters())
                    + list(block.attns[-1].parameters())
                    + new_alpha_params
                )
                optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
                grow_events.append((step, target_block, sum(model.n_attn_per_block)))
                next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

        # SinusoidalAlpha 의 amplitude/bias/log_freq 저장
        final_alphas = []
        for bi, block in enumerate(model.blocks):
            for ai, alpha_mod in enumerate(block.attn_alphas):
                final_alphas.append(
                    (
                        bi,
                        ai,
                        {
                            "amplitude": alpha_mod.amplitude.detach().cpu().clone(),
                            "bias": alpha_mod.bias.detach().cpu().clone(),
                            "log_freq": alpha_mod.log_freq.detach().cpu().clone(),
                        },
                    )
                )
        runs[(init_amp, seed)] = {
            "losses": losses,
            "grow_events": grow_events,
            "final_alphas": final_alphas,
        }
        n_last = min(100, len(losses))
        final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0
        print(f"  done: final_loss(last 100 avg)={final_loss:.4f}, n_added={len(grow_events)}")

        del model, optimizer, trigger, data_iter
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — init_amp x seed

|amplitude| mean: amplitude 가 init 에서 의미 있게 변했는가? (vanishing 해소 1차 지표)

In [ ]:
import math
import statistics

import numpy as np

from graphlm.neuron.positional_analysis import eval_positional_alpha_from_state

T_EVAL = BLOCK_SIZE

init_log_freqs = np.linspace(math.log(0.01), math.log(1.0), BACKBONE["hidden_dim"])

print(
    f"{'init_amp':>9}  {'seed':>5}  {'n_added':>7}  {'|amp| mean':>10}  "
    f"{'log_freq d':>10}  {'ch-dead%':>8}  {'final_loss':>11}"
)
print("-" * 85)
table = {}
for init_amp in INIT_AMPS:
    table[init_amp] = {}
    for seed in SEEDS:
        r = runs[(init_amp, seed)]
        added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
        n_added = len(added)
        if n_added == 0:
            amp_mean = math.nan
            lf_delta = math.nan
            ch_dead = math.nan
        else:
            amp_means = []
            lf_deltas = []
            ch_deads = []
            for _, _, a_dict in added:
                amp_means.append(float(np.abs(a_dict["amplitude"].numpy()).mean()))
                lf_deltas.append(float(np.abs(a_dict["log_freq"].numpy() - init_log_freqs).mean()))
                a_matrix = eval_positional_alpha_from_state(a_dict, T_EVAL)
                ch_deads.append(float((np.abs(a_matrix).max(axis=0) < DEAD_THRESHOLD).mean()))
            amp_mean = statistics.mean(amp_means)
            lf_delta = statistics.mean(lf_deltas)
            ch_dead = statistics.mean(ch_deads)

        n_last = min(100, len(r["losses"]))
        final_loss = sum(r["losses"][-n_last:]) / n_last if n_last > 0 else 0.0
        table[init_amp][seed] = dict(
            n_added=n_added,
            amp_mean=amp_mean,
            lf_delta=lf_delta,
            ch_dead=ch_dead,
            final_loss=final_loss,
        )
        if n_added == 0:
            amp_s, lf_s, ch_s = "    n/a ", "    n/a ", "    n/a "
        else:
            amp_s = f"{amp_mean:>10.4f}"
            lf_s = f"{lf_delta:>10.4f}"
            ch_s = f"{ch_dead:>8.1%}"
        print(
            f"{init_amp:>9.2f}  {seed:>5}  {n_added:>7d}  {amp_s}  {lf_s}  "
            f"{ch_s}  {final_loss:>11.4f}"
        )

## 6. init_amp 별 통계 + Phase 6 baseline 비교

In [ ]:
agg = {}
for init_amp in INIT_AMPS:
    fls = [table[init_amp][s]["final_loss"] for s in SEEDS]
    amps = [table[init_amp][s]["amp_mean"] for s in SEEDS]
    agg[init_amp] = dict(
        loss_mean=statistics.mean(fls),
        loss_sigma=statistics.stdev(fls) if len(fls) > 1 else 0,
        amp_mean=statistics.mean(amps),
    )

print(f"{'init_amp':>9}  {'final_loss mean':>16}  {'sigma':>7}  {'|amp| mean':>10}  verdict")
print("-" * 85)
for init_amp in INIT_AMPS:
    a = agg[init_amp]
    if init_amp == 0.0:
        verdict = "Phase 6 baseline (재현)"
    else:
        # amplitude 가 init_amp 보다 의미 있게 자랐는가? (vanishing 해소 여부)
        ratio = a["amp_mean"] / init_amp
        if a["amp_mean"] > init_amp * 1.5:
            verdict = f"[ok] amplitude 학습 활성 (final/init = {ratio:.2f}x)"
        elif a["amp_mean"] < init_amp * 0.7:
            verdict = f"[neg] amplitude 감소 (final/init = {ratio:.2f}x)"
        else:
            verdict = f"[neutral] amplitude 정체 (final/init = {ratio:.2f}x)"
    print(
        f"{init_amp:>9.2f}  {a['loss_mean']:>16.4f}  {a['loss_sigma']:>7.4f}  "
        f"{a['amp_mean']:>10.4f}  {verdict}"
    )

print()
print("=== Phase 6 baseline (init_amp=0.0) vs Phase 7 init_amp sweep ===")
print(f"  Phase 6 positional       : {PHASE6_POS_MEAN:.4f} +/- {PHASE6_POS_SIGMA:.4f}")
print(f"  Phase 7 init_amp=0.0     : {agg[0.0]['loss_mean']:.4f} +/- {agg[0.0]['loss_sigma']:.4f}")
print(f"  Phase 6 per_channel ref  : {PHASE6_PERCH_MEAN:.4f}")
for init_amp in [v for v in INIT_AMPS if v != 0.0]:
    diff = agg[0.0]["loss_mean"] - agg[init_amp]["loss_mean"]
    larger_sigma = max(agg[0.0]["loss_sigma"], agg[init_amp]["loss_sigma"])
    ratio = abs(diff) / larger_sigma if larger_sigma > 0 else float("inf")
    print(
        f"  init_amp={init_amp:.2f} vs 0.0: diff={diff:+.4f}, |diff|/sigma={ratio:.2f}"
        f" ({'우위' if diff > 0 else '열위' if diff < 0 else 'tie'})"
    )

## 7. 시각화 — final_loss + amplitude 진화 + alpha(t,c) heatmap

In [ ]:
import matplotlib.pyplot as plt

from graphlm.neuron.positional_analysis import eval_positional_alpha_from_state

fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.3])

# (top-row) final_loss bar — init_amp 별 3 seed
ax_top = fig.add_subplot(gs[0, :])
width = 0.25
x_seeds = list(range(len(SEEDS)))
colors_amp = {0.0: "#888888", 0.03: "#1f77b4", 0.10: "#2ca02c"}
all_final = [table[ia][s]["final_loss"] for ia in INIT_AMPS for s in SEEDS]
for i, init_amp in enumerate(INIT_AMPS):
    offsets = [x + (i - 1) * width for x in x_seeds]
    vals = [table[init_amp][s]["final_loss"] for s in SEEDS]
    ax_top.bar(
        offsets,
        vals,
        width=width,
        color=colors_amp[init_amp],
        label=f"init_amp={init_amp}",
        alpha=0.85,
    )
ax_top.axhline(
    PHASE6_POS_MEAN,
    color="#888888",
    linestyle=":",
    lw=1,
    label=f"Phase 6 positional ({PHASE6_POS_MEAN})",
)
ax_top.axhline(
    PHASE6_PERCH_MEAN,
    color="#ff7f0e",
    linestyle=":",
    lw=1,
    label=f"Phase 6 per_channel ({PHASE6_PERCH_MEAN})",
)
ax_top.axhline(
    GRAPHLM_PHASE1_4L_FINAL_LOSS,
    color="red",
    linestyle="--",
    lw=1,
    label=f"GraphLM 4L baseline ({GRAPHLM_PHASE1_4L_FINAL_LOSS})",
)
ax_top.set_xlabel("seed")
ax_top.set_xticks(x_seeds)
ax_top.set_xticklabels([str(s) for s in SEEDS])
ax_top.set_ylabel("final loss (last 100 avg)")
ax_top.set_title("Phase 7 final loss: init_amp sweep × seed")
ax_top.legend(loc="upper right", fontsize=8)
ax_top.grid(alpha=0.3, axis="y")
ymin_d = min(min(all_final), PHASE6_PERCH_MEAN, PHASE6_POS_MEAN)
ymax_d = max(max(all_final), GRAPHLM_PHASE1_4L_FINAL_LOSS)
margin = max(0.02, (ymax_d - ymin_d) * 0.15)
ax_top.set_ylim(ymin_d - margin, ymax_d + margin)

# (bottom row) seed=42 init_amp 별 첫 added attn 의 α(t,c) heatmap
for col, init_amp in enumerate(INIT_AMPS):
    ax = fig.add_subplot(gs[1, col])
    r = runs[(init_amp, SEEDS[0])]
    added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
    if added:
        bi0, _ai0, a_dict0 = added[0]
        a_mat = eval_positional_alpha_from_state(a_dict0, T_EVAL)
        vmax = max(np.abs(a_mat).max(), 0.1)
        im = ax.imshow(a_mat.T, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_xlabel("position t")
        if col == 0:
            ax.set_ylabel("channel c")
        ax.set_title(f"init_amp={init_amp}, seed={SEEDS[0]}, block={bi0}, added#0")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(OUT_DIR / "amplitude_smooth_start.png", dpi=120)
plt.show()

## 결과 요약 / Phase 8 권장 방향

확인 포인트:
- §5 |amp| mean 의 init_amp 별 진화 — init=0 (Phase 6 재현) 0.012 / init=0.03 0.??? / init=0.10 0.???
- §5 log_freq d — amplitude 활성화가 log_freq 학습도 견인했는가?
- §6 final_loss verdict — init_amp 별 sweet spot 존재? Phase 6 (init=0) 대비 개선?
- §7 heatmap — init_amp 클수록 position dependency 더 두드러진가?

**판정 시나리오**:
- **A. amplitude 학습 활성 + final_loss 개선** ⭐ — sinusoidal 가치 입증. Phase 8: MLP-based α(t) 비교 또는 task 확대
- **B. amplitude 학습 + final_loss 동일/열위** — sinusoidal 표현력 한계. Phase 8: paradigm 다른 axis (nn.Parameter 정적 shape 초월, 기존 Phase 7+ 후보 → Phase 8+)
- **C. amplitude 여전히 미학습** — smooth-start 도 부족. sinusoidal 자체 한계 확정 → axis 전환

**참고**:
- Phase 6 결과 (Notion): https://www.notion.so/36be8b70b7aa813e8511ed331bacd8db
- 차기 axis 후보: https://www.notion.so/36be8b70b7aa8142bebdc9706c64ed52